# Campi Flegrei: a caldera that rises

West of Naples, the ground of the Campi Flegrei caldera has been rising for years: at
Pozzuoli the harbour has lifted out of the water before (*bradyseism*). EGMS measures it
from space, millimetre by millimetre.

This notebook:
1. makes a **map** of vertical ground speed around Naples (L3, 100 m);
2. takes the **time series** of the fastest-rising cells and shows how much the ground
   rose, date by date.

**Downloads:** ~320 MB the first time (4 tiles for the map, one tile's time series), nothing
afterwards.

*Data: EGMS, © European Union, Copernicus Land Monitoring Service. Anything you publish from it must credit the source.*

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from pyproj import Transformer

from egms_client import EgmsArchive, Query

# The key comes from EGMS_SERVICE_KEY or EGMS_SERVICE_KEY_FILE (see the README).
# Downloads are kept here: run the notebook again and nothing is downloaded.
DATA = Path(os.environ.get("EGMS_DATA_DIR", "egms-data"))
RELEASE = "2020-2025"
to_laea = Transformer.from_crs("EPSG:4326", "EPSG:3035", always_xy=True)

## 1. The map

The L3 tiles a box touches, pasted on one 100 m grid in EPSG:3035.

In [ ]:
from egms_client.mosaic import lonlat_to_laea, mosaic, to_xarray

BOX = (13.95, 40.75, 14.55, 41.0)  # west, south, east, north in degrees
query = Query(levels=("L3",), release=RELEASE, bbox=BOX, product_type="ORTHO-UP")

async with EgmsArchive() as archive:
    found = [h for s in await archive.find(query) for h in s.hits]
    print(f"{len(found)} tiles, {sum(h.filesize for h in found) / 1e6:.0f} MB to download (once)")
    tiffs = await archive.fetch(query, DATA, keep=["tiff"])

minx, miny, maxx, maxy = lonlat_to_laea(*BOX)
grid = to_xarray(mosaic(tiffs, minx=minx, miny=miny, maxx=maxx, maxy=maxy))
grid

In [ ]:
POZZUOLI = to_laea.transform(14.12, 40.825)

fig, ax = plt.subplots(figsize=(10, 5.5))
grid.plot(
    ax=ax, cmap="RdBu_r", vmin=-60, vmax=60, cbar_kwargs={"label": "vertical speed (mm/year)"}
)
ax.annotate(
    "Pozzuoli",
    POZZUOLI,
    xytext=(POZZUOLI[0] - 9000, POZZUOLI[1] + 7000),
    arrowprops={"arrowstyle": "-"},
)
ax.set_aspect("equal")
ax.set_title("Vertical ground speed around Naples, 2020-2025 (red: rising)")
ax.set_axis_off()
print(f"fastest cell: {float(grid.max()):.0f} mm/year")

## 2. How fast, and since when

Each L3 tile also ships a CSV with the **displacement of every 100 m cell at every
acquisition date**. `keep=["parquet"]` turns it into GeoParquet (about a quarter of the CSV)
and drops the CSV. Pozzuoli is in tile **E46N19**.

In [ ]:
import pandas as pd

from egms_client.tables import to_xarray as series_to_xarray

tile = Query(levels=("L3",), release=RELEASE, tile=(46, 19), product_type="ORTHO-UP")
async with EgmsArchive() as archive:
    (parquet,) = await archive.fetch(tile, DATA, keep=["parquet"])

cells = pd.read_parquet(parquet).drop(columns="geometry")
near = cells[np.hypot(cells.easting - POZZUOLI[0], cells.northing - POZZUOLI[1]) < 3000]
fastest = near.nlargest(25, "mean_velocity")
series = series_to_xarray(fastest)
print(f"{len(cells):,} cells in the tile, {len(near):,} within 3 km of Pozzuoli")
fastest[["pid", "easting", "northing", "mean_velocity", "acceleration"]].head()

In [ ]:
curve = series.mean("pid") / 10  # mm to cm
risen = float(curve[-1] - curve[0])

fig, ax = plt.subplots(figsize=(10, 4))
curve.plot(ax=ax, color="#e34948")
ax.set_ylabel("uplift since the first date (cm)")
ax.set_xlabel("")
ax.set_title(
    f"The 25 fastest-rising cells near Pozzuoli: +{risen:.0f} cm in {len(curve)} acquisitions"
)
ax.grid(alpha=0.3)

The curve is close to a straight line, but **steeper in 2024-2025** than in 2021: the uplift
speeds up, which is what the `acceleration` column measures. The mean velocity alone would hide
it.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={"width_ratios": [1.3, 1]})
grid.plot(ax=left, cmap="RdBu_r", vmin=-60, vmax=60, cbar_kwargs={"label": "mm/year"})
left.set_aspect("equal")
left.set_axis_off()
left.set_title("Vertical speed around Naples")
curve.plot(ax=right, color="#e34948")
right.set_title(f"Uplift near Pozzuoli: +{risen:.0f} cm")
right.set_ylabel("cm")
right.set_xlabel("")
right.grid(alpha=0.3)
fig.text(
    0.01,
    0.01,
    "EGMS Ortho L3 2020-2025. Contains modified Copernicus Land Monitoring Service information.",
    fontsize=8,
)
fig.savefig("../docs/images/notebook-campi-flegrei.png", dpi=100, bbox_inches="tight")